In [ ]:
## Basic
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
## Excel
import os
#import xlwt
#from openpyxl import load_workbook
## Data process
from scipy.interpolate import interp1d #Interpolation
from scipy.signal import savgol_filter #Smooth
import math
import time
from sklearn.linear_model import LinearRegression
from datetime import datetime,timedelta

#import cv2

import random
from IPython import display
from math import dist
from scipy.optimize import curve_fit,fmin
import struct
from scipy.optimize import minimize
from scipy.signal import savgol_filter #Smooth


In [ ]:
cm =1/2.54
def fig_pre_def(fx=8, fy=6,lw=1,dpi = 200):
    cm =1/2.54
    plt.rcParams['figure.dpi'] = dpi
    plt.rc('font', family='Helvetica')         # Font family
    plt.rcParams['axes.linewidth'] = lw     # Width of the axes frame
    return  plt.figure(figsize=(fx*cm,fy*cm))   
    ax = fig.add_subplot(1, 1, 1)
def fig_post_def(xticks =(None,None),yticks =(None,None), # X-axis range and tick spacing
                 xlim=(None,None),ylim=(None,None), # Y-axis range and tick spacing
                 xlabel = None,ylabel=None,# Labels of the X and Y axes
                 title=None,ncol=1, # Legend title and number of columns 
                 lg_fs = 6,lb_fs = 8,# Font sizes of the legend and the labels
                 unit = 1/2.54, SVG = False,onefig=True # Units converted to inches because dpi affects the scaling
                 ):
    if xticks[0] != None:
        # Edit the major and minor tick locations; if not specified, they are set automatically
        ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(xticks[0]))    
        ax.xaxis.set_minor_locator(mpl.ticker.MultipleLocator(xticks[1]))    
    if yticks[0] != None:
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(yticks[0]))
        ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(yticks[1]))
    #plt.rc('xtick', labelsize=fontsize)          # Font size of the X-axis tick labels
    #plt.rc('ytick', labelsize=fontsize)          # Font size of the Y-axis tick labels
    ax.tick_params(axis='x', labelsize= lb_fs)
    ax.tick_params(axis='y', labelsize= lb_fs)
    ax.xaxis.set_tick_params(which='major', size=10*unit, width=1, direction='in', top='on')    # X-axis major ticks
    ax.xaxis.set_tick_params(which='minor', size=7*unit, width=1, direction='in', top='on')     # X-axis minor ticks
    ax.yaxis.set_tick_params(which='major', size=10*unit, width=1, direction='in', right='on')   # Y-axis major ticks
    ax.yaxis.set_tick_params(which='minor', size=7*unit, width=1, direction='in', right='on')   # Y-axis minor ticks
    new_rc_params = {'text.usetex': False,
        "svg.fonttype": 'none'
        }
    plt.rcParams['axes.linewidth'] = 1     # Width of the axes frame
    mpl.rcParams.update(new_rc_params)
    if xlim != None:
        ax.set_xlim(xlim)
    if ylim != None:
        ax.set_ylim(ylim)
    if xlabel != None:
        ax.set_xlabel(xlabel,fontsize=lb_fs)
    if ylabel != None:
        ax.set_ylabel(ylabel,fontsize=lb_fs)

    if SVG == True:
        ax.axes.xaxis.set_ticklabels([])
        ax.axes.yaxis.set_ticklabels([])

        new_rc_params = {'text.usetex': False,
        "svg.fonttype": 'none'
        }
        plt.rcParams['axes.linewidth'] = 1     # Width of the axes frame
        mpl.rcParams.update(new_rc_params)
    if onefig ==True:
        plt.gca().set_position([0, 0, 1, 1])


    plt.legend(fontsize=lg_fs,title=title,title_fontsize=lg_fs,frameon= False,ncol=ncol)
    plt.rcParams['figure.dpi'] = 80
    
#-----Use the following commands for subsequent plotting-----
#fig = fig_pre_def(7.5, 4,1)# Figure size for publication; 7.5 cm x 4 cm, line width = 1
#ax = fig.add_subplot(1, 1, 1)          # Create a 1x1 subplot grid; use the first cell
#plt.plot(x,y     ,'--ob',markersize=8*cm,linewidth=3*cm,mew=0,label='point source')
#fig_post_def()
#--------------------------

def sorting(input_data, bin_number , lower_range, upper_range):
    hist, edges = np.histogram( # hist: counts in each bin; edges: boundaries of each bin
    input_data, # Array to be analyzed
    bins=bin_number, 
    range=(lower_range, upper_range), # Range
    density=False) # Whether to normalize into fractions
    Bin_center = []
    for i in range(len(edges)-1):
        Bin_center_single = round((edges[i]+edges[i+1])/2,2)
        Bin_center = np.append(Bin_center,Bin_center_single)
    
    return Bin_center, hist
def interpolation_function(data_x,data_y,new_x):
    new_y = interp1d(data_x, data_y,fill_value="extrapolate")(new_x)
    return new_y

In [ ]:
from scipy.optimize import curve_fit,fmin
def gauss_wo_baseline(x,  A, x0, sigma):
    return A * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))
def gauss_fit_wo_baseline(x, y):  
    mean = sum(x * y) / sum(y)
    sigma = np.sqrt(sum(y * (x - mean) ** 2) / sum(y))
    try:  
        popt, pcov = curve_fit(gauss_wo_baseline, x, y, p0=[1, mean, sigma], maxfev=1000)
        return popt
    except RuntimeError:
        print("Error - curve_fit failed")
        return  [0,mean,sigma]